In [12]:
import os
import json
import re
import argparse
from datasets import load_dataset
from datetime import datetime
from tqdm import tqdm

def robust_json_loads(data):
    """Parse potentially malformed JSON strings (trailing commas, control chars)."""
    if not isinstance(data, str) or not data:
        return data
    try:
        return json.loads(data)
    except json.JSONDecodeError:
        try:
            fixed = re.sub(r',\s*([\]}])', r'\1', data)
            fixed = re.sub(r'[\x00-\x1F\x7F]', '', fixed)
            return json.loads(fixed)
        except Exception:
            return {}


def parse_timestamp(ts_str):
    """Parse an ISO timestamp string; return datetime.max on failure."""
    if not ts_str:
        return datetime.max
    try:
        return datetime.fromisoformat(ts_str.replace('Z', '+00:00'))
    except Exception:
        return datetime.max


# ---------------------------------------------------------------------------
# Span traversal & content extraction
# ---------------------------------------------------------------------------

def collect_all_spans(spans, container):
    """Recursively collect every span node from a nested span tree."""
    if not spans:
        return
    for span in spans:
        container.append(span)
        if span.get("child_spans"):
            collect_all_spans(span["child_spans"], container)


def extract_step_content(span):
    """
    Pull the meaningful text out of a span's logs and attributes.
    Returns a single concatenated string with no length limit.
    """
    parts = []

    for log in span.get('logs', []):
        body = log.get('body', {})
        if body:
            # parts.append(json.dumps(body, ensure_ascii=False))
            parts.append(body)

    attrs = span.get('span_attributes', {})
    for key in ['input.value', 'output.value', 'llm.output_messages.0.message.content']:
        if attrs.get(key):
            # parts.append(str(attrs[key]))
            parts.append(attrs.get(key))

    # return " ".join(parts)
    return parts[-1]


# ---------------------------------------------------------------------------
# Core per-entry processor
# ---------------------------------------------------------------------------

def process_entry(entry):
    """
    Parse a single TRAIL entry and return structured per-step data.

    Returns a dict with:
        trace_id   : str
        steps      : list of {"span_id", "span_name", "text", "label"}
        found      : bool  (False if the entry had no usable spans)
        debug      : dict  (diagnostic info)
    """
    trace_data = robust_json_loads(entry.get('trace', {}))
    label_data = robust_json_loads(entry.get('labels', {}))
    trace_id   = trace_data.get('trace_id', 'unknown')

    # --- Flatten span tree --------------------------------------------------
    all_spans = []
    collect_all_spans(trace_data.get('spans', []), all_spans)

    if not all_spans:
        return {
            "trace_id": trace_id, "steps": [],
            "found": False, "debug": {"reason": "no spans found", "raw_count": 0}
        }

    # --- Filter to meaningful steps -----------------------------------------
    SKIP_NAMES = {'main', 'create_agent_hierarchy'}
    # valid_steps = [
    #     s for s in all_spans
    #     if s.get('span_name') not in SKIP_NAMES
    #     and len(extract_step_content(s).strip()) > 2
    # ]
    valid_steps = all_spans

    debug_note = "normal"
    if not valid_steps:
        debug_note = "fallback: no high-quality steps; using all non-main spans"
        valid_steps = [s for s in all_spans if s.get('span_name') not in SKIP_NAMES]
    if not valid_steps:
        debug_note = "extreme fallback: only main/wrapper nodes available"
        valid_steps = all_spans

    # --- Determine root-cause span ID ---------------------------------------
    span_lookup = {
        str(s.get('span_id', '')).strip().lower(): s
        for s in all_spans if s.get('span_id')
    }

    root_cause_id = None
    errors = label_data.get('errors', [])

    if errors:
        matched = []
        for err in errors:
            loc = str(err.get('location', '')).strip().lower()
            if loc in span_lookup:
                ts = parse_timestamp(span_lookup[loc].get('timestamp'))
                matched.append((loc, ts))

        if matched:
            # Pick the earliest-occurring error as the root cause
            matched.sort(key=lambda x: x[1])
            root_cause_id = matched[0][0]
        else:
            # Fallback: label the earliest valid step as root cause
            valid_steps_sorted = sorted(
                valid_steps, key=lambda s: parse_timestamp(s.get('timestamp'))
            )
            root_cause_id = str(valid_steps_sorted[0].get('span_id', '')).strip().lower()
            debug_note += " | label fallback: earliest step used as root cause"

    # --- Build step records -------------------------------------------------
    steps = []
    for span in valid_steps:
        span_id = str(span.get('span_id', '')).strip().lower()
        steps.append({
            "span_id":   span_id,
            "span_name": span.get('span_name', 'unknown'),
            # "text":      f"Step: {span.get('span_name', 'step')}\n{extract_step_content(span)}",
            "text":     extract_step_content(span),
            "label":     1 if span_id == root_cause_id else 0,
        })

    found = bool(root_cause_id and steps)
    return {
        "trace_id": trace_id,
        "steps":    steps,
        "found":    found,
        "debug":    {"reason": debug_note, "raw_count": len(all_spans)},
    }

In [13]:
def run_processing(
    dataset_name="PatronusAI/TRAIL",
    save_dir="./processed_data/trajectories/",
    split="gaia",
):
    import os, json
    from datasets import load_dataset
    from tqdm import tqdm

    os.makedirs(save_dir, exist_ok=True)
    ds = load_dataset(dataset_name, split=split)
    print(f"Loaded {len(ds)} trajectories from '{split}' split.")

    failed, saved = [], 0

    for entry in tqdm(ds, desc="Processing trajectories"):
        result = process_entry(entry)
        if not result["found"]:
            failed.append({"trace_id": result["trace_id"], "info": result["debug"]})
            continue
        out_path = os.path.join(save_dir, f"{result['trace_id']}.json")
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        saved += 1

    total_steps, total_positives = 0, 0
    for fname in os.listdir(save_dir):
        if not fname.endswith('.json'):
            continue
        with open(os.path.join(save_dir, fname)) as f:
            data = json.load(f)
        total_steps     += len(data['steps'])
        total_positives += sum(s['label'] for s in data['steps'])

    print(f"\nDone. {saved} saved / {len(failed)} failed")
    print(f"Total steps: {total_steps} | Root-cause steps: {total_positives}")
    print(f"Output: {save_dir}")
    if failed:
        print("\nFailed trajectories:")
        for item in failed:
            print(f"  {item['trace_id']} — {item['info']['reason']}")

# run it
run_processing(split="gaia")

Loaded 117 trajectories from 'gaia' split.


Processing trajectories: 100%|██████████| 117/117 [00:01<00:00, 89.85it/s] 



Done. 114 saved / 3 failed
Total steps: 3534 | Root-cause steps: 114
Output: ./processed_data/trajectories/

Failed trajectories:
  d2868d12880a41ad5ed1fb3bb39159d5 — normal
  f510c80d120dc75e4259704184ee802d — normal
  fa31e4af04a2469c88d6e8845e8aac69 — normal


In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
from attribscope.data.trajectory import load_dataset

In [11]:
algo = load_dataset(
    path="data/ww", subset="algorithm-generated"
)

## Refactor

In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
from attribscope.svd2.computation import (
    fit_all,
    score_all
)
from attribscope.svd2.utils import (
    load_representations,
    _resolve_model_tag
)
import pandas as pd
from pathlib import Path
import torch

In [ ]:
# for model, subset, (rep, pooling), (loss, temperature), mode \
#     in tqdm(combos, total=total):


In [3]:
outputs_root = Path("/data/hoang/attrib")
rep    = "grads"
model  = "qwen3-8b"
# subset = "hand-crafted"
subset = "algorithm-generated"
loss   = "ntp"
temperature = 1.6
data_dir = Path("data/ww")
device = torch.device("cuda:1")

model_tag = _resolve_model_tag(model, loss, temperature)
print(f"model_tag: {model_tag}")
rep_dir = Path(model_tag) / "reps" / subset
base_dir = outputs_root / rep

print(base_dir)
print(rep_dir)

model_tag: qwen3-8b
/data/hoang/attrib/grads
qwen3-8b/reps/algorithm-generated


In [4]:
reps = load_representations(
    base_dir=base_dir,
    rep_dir=rep_dir,
    subset=subset,
    pooling="grad",
    weight_names="all",
    data_dir=data_dir / subset,
    device=device,
    files=None
)

Loading [algorithm-generated] at /data/hoang/attrib/grads/qwen3-8b/reps/algorithm-generated: 100%|██████████| 126/126 [00:24<00:00,  5.21it/s]


In [ ]:
from pathlib import Path
import torch
from attribscope.svd2.computation import run_pipeline
from attribscope.svd2.utils import load_representations, _resolve_model_tag

In [2]:

# ── Config ───────────────────────────────────────────────────────────────────
loss       = "ntp"
subset     = "hand-crafted"
model      = "llama-3.1-8b"
model_tag  = _resolve_model_tag(model, loss, temperature=1.0)
base_dir   = Path("/data/hoang/attrib")
rep_dir    = Path(f"grads/{model}/reps/{subset}")
data_dir   = Path("data/ww")
device     = torch.device("cuda:7")

n_components_fit   = 10
n_components_score = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
ks                 = [1, 3, 5, 10]

# load
load_kwargs = dict(
    weight_names="all", 
    base_dir=base_dir, 
    rep_dir=rep_dir,
    device=device,
)

val_reps  = load_representations(
    **load_kwargs, 
    subset=subset, 
    pooling="grad", 
    data_dir=data_dir / subset
)


Loading [hand-crafted] at /data/hoang/attrib/grads/llama-3.1-8b/reps/hand-crafted: 100%|██████████| 58/58 [00:40<00:00,  1.45it/s]


In [3]:
# run 
val_df, test_df = run_pipeline(
    val_reps=val_reps, test_reps=val_reps,
    mode="A",                        # fit on val, score both
    n_components_fit=n_components_fit,
    n_components_score=n_components_score,
    ks=ks, device=device,
)

SVD fit: 100%|██████████| 291/291 [00:35<00:00,  8.12it/s]


In [ ]:
test_reps = load_representations(
    **load_kwargs, 
    subset="hand-crafted", 
    pooling=["grad"], 
    data_dir=data_dir / "hand-crafted"
)

# run 
val_df, test_df = run_pipeline(
    val_reps=val_reps, test_reps=test_reps,
    mode="A",                        # fit on val, score both
    n_components_fit=n_components_fit,
    n_components_score=n_components_score,
    ks=ks, device=device,
)

# inspect
test_df.groupby(["method", "c", "centered", "direction", "k"])["step_acc"].mean().unstack("k")

In [ ]:
# safetensors_dir = base_dir / rep_dir
# files = sorted(safetensors_dir.glob("*.safetensors"), 
#                key=lambda x: int(x.stem))

# val_files, test_files = split_data(files, ratio=0.5, seed=42)

In [ ]:
def run_pipeline(
    model:  str,
    fit_set: str,
    score_set: str,
    loss:   str,   # "ntp" | "kl_uniform" | "kl_temp"
    rep:    str,   # "hidden" | "grads"
    n_components:       int              = 10,
    n_components_score: list[int] | str  = "all",
    ks:                 list[int]        = (1, 3, 5, 10),
    pooling:            list[str] | None = None,
    temperature:        float | None     = None,
    outputs_root:       Path             = Path("/data/hoang/attrib"),
    data_dir:           Path             = Path("data/ww"),
    device:             str | None       = None,
) -> pd.DataFrame:
    pass

In [ ]:
    # outputs_root = Path("/data/hoang/attrib")
    # rep    = "grads"
    # model  = "qwen3-8b"
    # subset = "algorithm-generated"
    # loss   = "ntp"
    # temperature = 1.6

    # pooling = "grad"
    # weight_names = "all"

    # device = torch.device("cuda:7")